In [0]:
# ===========================================
# Notebook Name:
# 01_ingest_event_results
#
# Purpose:
# Fetch tournament result API responses for
# multiple tournaments selected by the Ops
# fetch-target table.
#
# Input:
# pokemon.ops.result_fetch_targets
#
# Outputs:
# pokemon.bronze.event_result_raw
# pokemon.bronze.scrape_log
#
# Bronze grain:
# One row per tournament_id and response
# version.
#
# Idempotency:
# tournament_id + response_hash
# ===========================================

In [0]:
from datetime import datetime, timezone
import hashlib
import json
import time
import uuid

import requests

from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry

from pyspark.sql import functions as F
from pyspark.sql.types import (
    DateType,
    IntegerType,
    LongType,
    StringType,
    StructField,
    StructType,
    TimestampType,
)

In [0]:
RESULT_FETCH_TARGETS_TABLE = (
    "pokemon.ops.result_fetch_targets"
)

EVENT_RESULT_RAW_TABLE = (
    "pokemon.bronze.event_result_raw"
)

SCRAPE_LOG_TABLE = (
    "pokemon.bronze.scrape_log"
)

RESULT_API_URL_TEMPLATE = (
    "既存Notebookで使用している大会結果API URL"
)
RESULT_API_URL_TEMPLATE = (
    "https://players.pokemon-card.com/"
    "event_result/{tournament_id}"
)

REQUEST_TIMEOUT_SECONDS = 30
REQUEST_INTERVAL_SECONDS = 1.0

MAX_FETCH_TARGETS = 500

FAIL_NOTEBOOK_IF_ANY_ERROR = True

In [0]:
if not spark.catalog.tableExists(
    RESULT_FETCH_TARGETS_TABLE
):
    raise ValueError(
        "Result fetch target table does not exist: "
        f"{RESULT_FETCH_TARGETS_TABLE}"
    )

targets_source_df = spark.table(
    RESULT_FETCH_TARGETS_TABLE
)

target_columns = {
    field.name
    for field in targets_source_df.schema.fields
}

if "tournament_id" not in target_columns:
    raise ValueError(
        "result_fetch_targets does not contain "
        "tournament_id"
    )

target_select_columns = [
    F.col("tournament_id")
    .cast("int")
    .alias("tournament_id"),
]

if "fetch_reason" in target_columns:
    target_select_columns.append(
        F.col("fetch_reason")
    )
else:
    target_select_columns.append(
        F.lit("unknown")
        .alias("fetch_reason")
    )

if "priority" in target_columns:
    target_select_columns.append(
        F.col("priority")
        .cast("int")
        .alias("priority")
    )
else:
    target_select_columns.append(
        F.lit(99)
        .alias("priority")
    )

if "event_date" in target_columns:
    target_select_columns.append(
        F.col("event_date")
    )
else:
    target_select_columns.append(
        F.lit(None)
        .cast("date")
        .alias("event_date")
    )

if "event_title" in target_columns:
    target_select_columns.append(
        F.col("event_title")
    )
else:
    target_select_columns.append(
        F.lit(None)
        .cast("string")
        .alias("event_title")
    )

targets_df = (
    targets_source_df
    .select(
        *target_select_columns
    )
    .filter(
        F.col("tournament_id").isNotNull()
    )
    .dropDuplicates(
        ["tournament_id"]
    )
    .orderBy(
        F.col("priority").asc(),
        F.col("event_date").desc_nulls_last(),
        F.col("tournament_id").desc(),
    )
    .limit(
        MAX_FETCH_TARGETS
    )
)

target_count = targets_df.count()

print(
    "Tournament result fetch targets:",
    target_count,
)

display(targets_df)

if target_count == 0:
    print(
        "No tournament result fetch targets."
    )

In [0]:
retry_strategy = Retry(
    total=3,
    connect=3,
    read=3,
    status=3,
    backoff_factor=1.0,
    status_forcelist=[
        429,
        500,
        502,
        503,
        504,
    ],
    allowed_methods=[
        "GET",
    ],
)

http_adapter = HTTPAdapter(
    max_retries=retry_strategy
)

session = requests.Session()

session.mount(
    "https://",
    http_adapter,
)

session.mount(
    "http://",
    http_adapter,
)

REQUEST_HEADERS = {
    "Accept": (
        "application/json, "
        "text/plain, */*"
    ),
    "Accept-Language": (
        "ja,en-US;q=0.9,en;q=0.8"
    ),
    "User-Agent": (
        "Mozilla/5.0 "
        "(Macintosh; Intel Mac OS X 10_15_7) "
        "AppleWebKit/537.36 "
        "(KHTML, like Gecko) "
        "Chrome/150.0.0.0 "
        "Safari/537.36"
    ),
    "X-Accept-Version": "v1",
}

RESULT_REFERER_TEMPLATE = (
    "https://players.pokemon-card.com/"
    "event/result/detail/{tournament_id}"
)

In [0]:
def serialize_payload(
    payload_object,
) -> str:
    return json.dumps(
        payload_object,
        ensure_ascii=False,
        sort_keys=True,
        separators=(",", ":"),
    )
def calculate_sha256(
    value: str,
) -> str:
    return hashlib.sha256(
        value.encode("utf-8")
    ).hexdigest()

In [0]:
RESULT_LIST_KEYS = [
    "result",
    "results",
    "event_result",
    "event_results",
    "ranking",
    "rankings",
    "players",
]

def find_result_list(
    payload_object,
):
    if not isinstance(
        payload_object,
        dict,
    ):
        return []

    for key in RESULT_LIST_KEYS:
        value = payload_object.get(key)

        if isinstance(value, list):
            return value

    data = payload_object.get("data")

    if isinstance(data, dict):
        for key in RESULT_LIST_KEYS:
            value = data.get(key)

            if isinstance(value, list):
                return value

    return []

In [0]:
def build_result_api_url(
    tournament_id: int,
) -> str:
    return RESULT_API_URL_TEMPLATE.format(
        tournament_id=tournament_id
    )

In [0]:
def fetch_event_result(
    tournament_id: int,
):
    request_url = build_result_api_url(
        tournament_id
    )

    headers = dict(
        REQUEST_HEADERS
    )

    headers["Referer"] = (
        RESULT_REFERER_TEMPLATE.format(
            tournament_id=tournament_id
        )
    )

    started_at = time.perf_counter()

    response = session.get(
        request_url,
        headers=headers,
        timeout=REQUEST_TIMEOUT_SECONDS,
    )

    elapsed_ms = int(
        (
            time.perf_counter()
            - started_at
        ) * 1000
    )

    response.raise_for_status()

    content_type = (
        response.headers
        .get(
            "Content-Type",
            "",
        )
        .lower()
    )

    if "application/json" not in content_type:
        raise ValueError(
            "Tournament result API did not "
            "return JSON. "
            f"tournament_id={tournament_id}, "
            f"status={response.status_code}, "
            f"content_type={content_type}, "
            f"body={response.text[:500]}"
        )

    try:
        payload_object = response.json()

    except ValueError as error:
        raise ValueError(
            "Tournament result response "
            "could not be parsed as JSON. "
            f"tournament_id={tournament_id}, "
            f"body={response.text[:500]}"
        ) from error

    api_code = (
        payload_object.get("code")
        if isinstance(
            payload_object,
            dict,
        )
        else None
    )

    if (
        api_code is not None
        and int(api_code) != 200
    ):
        raise ValueError(
            "Tournament result API returned "
            "a non-success code. "
            f"tournament_id={tournament_id}, "
            f"api_code={api_code}"
        )

    result_items = find_result_list(
        payload_object
    )

    payload = serialize_payload(
        payload_object
    )

    response_hash = calculate_sha256(
        payload
    )

    return {
        "request_url": response.url,
        "response_status": (
            response.status_code
        ),
        "api_code": (
            int(api_code)
            if api_code is not None
            else None
        ),
        "payload_object": payload_object,
        "payload": payload,
        "response_hash": response_hash,
        "result_count": len(
            result_items
        ),
        "elapsed_ms": elapsed_ms,
    }

In [0]:
if not spark.catalog.tableExists(
    EVENT_RESULT_RAW_TABLE
):
    raise ValueError(
        "Existing event result raw table "
        "does not exist: "
        f"{EVENT_RESULT_RAW_TABLE}"
    )

event_raw_columns = {
    field.name
    for field in (
        spark.table(
            EVENT_RESULT_RAW_TABLE
        ).schema.fields
    )
}

print(
    "event_result_raw columns:",
    sorted(event_raw_columns),
)

existing_result_versions = set()

existing_raw_df = spark.table(
    EVENT_RESULT_RAW_TABLE
)

if {
    "tournament_id",
    "response_hash",
}.issubset(
    event_raw_columns
):
    existing_result_versions = {
        (
            int(row["tournament_id"]),
            row["response_hash"],
        )
        for row in (
            existing_raw_df
            .select(
                "tournament_id",
                "response_hash",
            )
            .filter(
                F.col(
                    "tournament_id"
                ).isNotNull()
                & F.col(
                    "response_hash"
                ).isNotNull()
            )
            .distinct()
            .collect()
        )
    }

print(
    "Existing result response versions:",
    len(existing_result_versions),
)

In [0]:
run_id = str(
    uuid.uuid4()
)

run_started_at = datetime.now(
    timezone.utc
)

raw_records = []
log_records = []

seen_result_versions_in_run = set()

success_count = 0
success_empty_count = 0
skipped_count = 0
error_count = 0

print(
    "Run ID:",
    run_id,
)



In [0]:
def build_raw_record(
    *,
    tournament_id,
    result,
    fetched_at,
):
    base_values = {
        "run_id": run_id,
        "tournament_id": tournament_id,
        "event_id": tournament_id,
        "payload": result["payload"],
        "response_hash": (
            result["response_hash"]
        ),
        "response_status": (
            result["response_status"]
        ),
        "http_status": (
            result["response_status"]
        ),
        "api_code": result["api_code"],
        "result_count": (
            result["result_count"]
        ),
        "source_url": (
            result["request_url"]
        ),
        "request_url": (
            result["request_url"]
        ),
        "scraped_at": fetched_at,
        "fetched_at": fetched_at,
        "ingestion_date": (
            fetched_at.date()
        ),
    }

    return {
        column_name: base_values.get(
            column_name
        )
        for column_name
        in event_raw_columns
    }

In [0]:
from pyspark.sql.types import (
    BooleanType,
    DateType,
    DoubleType,
    FloatType,
    IntegerType,
    LongType,
    ShortType,
    StringType,
    TimestampType,
)
scrape_log_schema = (
    spark.table(
        SCRAPE_LOG_TABLE
    ).schema
)
def default_value_for_type(
    data_type,
    current_time,
):
    if isinstance(
        data_type,
        StringType,
    ):
        return ""

    if isinstance(
        data_type,
        (
            IntegerType,
            LongType,
            ShortType,
        ),
    ):
        return 0

    if isinstance(
        data_type,
        (
            DoubleType,
            FloatType,
        ),
    ):
        return 0.0

    if isinstance(
        data_type,
        BooleanType,
    ):
        return False

    if isinstance(
        data_type,
        TimestampType,
    ):
        return current_time

    if isinstance(
        data_type,
        DateType,
    ):
        return current_time.date()

    return None

def build_log_record(
    *,
    tournament_id,
    fetch_reason,
    status,
    http_status,
    elapsed_ms,
    error_message,
    scraped_at,
    response_hash=None,
    result_count=None,
):
    base_values = {
        "run_id": str(run_id),
        "source_type": "event_result",
        "source_id": str(
            tournament_id
        ),
        "tournament_id": int(
            tournament_id
        ),
        "fetch_reason": (
            str(fetch_reason)
            if fetch_reason is not None
            else "unknown"
        ),
        "status": (
            str(status)
            if status is not None
            else "unknown"
        ),
        "http_status": (
            int(http_status)
            if http_status is not None
            else 0
        ),
        "response_status": (
            int(http_status)
            if http_status is not None
            else 0
        ),
        "elapsed_ms": (
            int(elapsed_ms)
            if elapsed_ms is not None
            else 0
        ),
        "error": (
            str(error_message)
            if error_message
            else ""
        ),
        "error_message": (
            str(error_message)
            if error_message
            else ""
        ),
        "response_hash": (
            str(response_hash)
            if response_hash
            else ""
        ),
        "result_count": (
            int(result_count)
            if result_count is not None
            else 0
        ),
        "scraped_at": scraped_at,
        "created_at": scraped_at,
        "updated_at": scraped_at,
    }

    output = {}

    for field in scrape_log_schema.fields:
        value = base_values.get(
            field.name
        )

        # StringTypeにはnullableでもNoneを入れない
        if (
            value is None
            and isinstance(
                field.dataType,
                StringType,
            )
        ):
            value = ""

        # NOT NULL列には型別デフォルト
        if (
            value is None
            and not field.nullable
        ):
            value = default_value_for_type(
                field.dataType,
                scraped_at,
            )

        output[field.name] = value

    return output

In [0]:
target_rows = targets_df.collect()

for index, target in enumerate(
    target_rows,
    start=1,
):
    tournament_id = int(
        target["tournament_id"]
    )

    fetch_reason = (
        target["fetch_reason"]
        or "unknown"
    )

    fetched_at = datetime.now(
        timezone.utc
    )

    print(
        f"[{index}/{target_count}] "
        f"tournament_id={tournament_id}, "
        f"reason={fetch_reason}"
    )

    try:
        result = fetch_event_result(
            tournament_id
        )

        result_version = (
            tournament_id,
            result["response_hash"],
        )

        is_new_response = (
            result_version
            not in existing_result_versions
            and result_version
            not in seen_result_versions_in_run
        )

        # Bronzeへ実際に追加したかどうか
        was_inserted = False

        if is_new_response:
            raw_record = build_raw_record(
                tournament_id=tournament_id,
                result=result,
                fetched_at=fetched_at,
            )

            raw_records.append(
                raw_record
            )

            seen_result_versions_in_run.add(
                result_version
            )

            was_inserted = True

        if result["result_count"] == 0:
            request_status = (
                "success_empty"
            )
            success_empty_count += 1

        elif is_new_response:
            request_status = "success"
            success_count += 1

        else:
            request_status = (
                "skipped_unchanged"
            )
            skipped_count += 1

        log_record = build_log_record(
            tournament_id=tournament_id,
            fetch_reason=fetch_reason,
            status=request_status,
            http_status=result[
                "response_status"
            ],
            elapsed_ms=result[
                "elapsed_ms"
            ],
            error_message="",
            scraped_at=fetched_at,
            response_hash=result[
                "response_hash"
            ],
            result_count=result[
                "result_count"
            ],
        )

        log_records.append(
            log_record
        )

        print(
            f"  status={request_status}, "
            f"result_count="
            f"{result['result_count']}, "
            f"new_response={is_new_response}, "
            f"inserted={was_inserted}"
        )

    except Exception as error:
        error_count += 1

        error_message = (
            f"{type(error).__name__}: "
            f"{str(error)}"
        )[:4000]

        log_record = build_log_record(
            tournament_id=tournament_id,
            fetch_reason=fetch_reason,
            status="error",
            http_status=0,
            elapsed_ms=0,
            error_message=error_message,
            scraped_at=fetched_at,
            response_hash="",
            result_count=0,
        )

        log_records.append(
            log_record
        )

        print(
            "  status=error, "
            f"error={error_message}"
        )

    if index < target_count:
        time.sleep(
            REQUEST_INTERVAL_SECONDS
        )

In [0]:
target_rows = targets_df.collect()

for index, target in enumerate(
    target_rows,
    start=1,
):
    tournament_id = int(
        target["tournament_id"]
    )

    fetch_reason = (
        target["fetch_reason"]
    )

    fetched_at = datetime.now(
        timezone.utc
    )

    print(
        f"[{index}/{target_count}] "
        f"tournament_id={tournament_id}, "
        f"reason={fetch_reason}"
    )

    try:
        result = fetch_event_result(
            tournament_id
        )

        result_version = (
            tournament_id,
            result["response_hash"],
        )

        is_new_response = (
            result_version
            not in existing_result_versions
            and result_version
            not in seen_result_versions_in_run
        )

        if result["result_count"] == 0:
            request_status = (
                "success_empty"
            )

            success_empty_count += 1

        elif not is_new_response:
            request_status = (
                "skipped_unchanged"
            )

            skipped_count += 1

        else:
            request_status = "success"
            success_count += 1

        if is_new_response:
            raw_records.append(
                build_raw_record(
                    tournament_id=(
                        tournament_id
                    ),
                    result=result,
                    fetched_at=fetched_at,
                )
            )

            seen_result_versions_in_run.add(
                result_version
            )

        log_records.append(
            build_log_record(
                tournament_id=(
                    tournament_id
                ),
                fetch_reason=fetch_reason,
                status=request_status,
                http_status=result[
                    "response_status"
                ],
                elapsed_ms=result[
                    "elapsed_ms"
                ],
                error_message=None,
                scraped_at=fetched_at,
                response_hash=result[
                    "response_hash"
                ],
                result_count=result[
                    "result_count"
                ],
            )
        )

        print(
            "  status="
            f"{request_status}, "
            "result_count="
            f"{result['result_count']}, "
            "new_response="
            f"{is_new_response}"
        )

    except Exception as error:
        error_count += 1

        error_message = (
            f"{type(error).__name__}: "
            f"{str(error)}"
        )[:4000]

        log_records.append(
            build_log_record(
                tournament_id=(
                    tournament_id
                ),
                fetch_reason=fetch_reason,
                status="error",
                http_status=None,
                elapsed_ms=None,
                error_message=(
                    error_message
                ),
                scraped_at=fetched_at,
            )
        )

        print(
            "  status=error, "
            f"error={error_message}"
        )

    if index < target_count:
        time.sleep(
            REQUEST_INTERVAL_SECONDS
        )

In [0]:
# ===========================================
# Cell 16:
# Validate and save scrape logs
# ===========================================

from pyspark.sql.types import StringType


if not log_records:
    print(
        "No scrape log rows to insert."
    )

else:
    # ---------------------------------------
    # Validate records before DataFrame creation
    # ---------------------------------------
    invalid_log_values = []

    for row_index, record in enumerate(
        log_records
    ):
        for field in scrape_log_schema.fields:
            value = record.get(
                field.name
            )

            # StringTypeへNoneを渡すと
            # PySparkValueErrorになる環境への対策
            if (
                isinstance(
                    field.dataType,
                    StringType,
                )
                and value is None
            ):
                invalid_log_values.append({
                    "row_index": row_index,
                    "column_name": field.name,
                    "data_type": str(
                        field.dataType
                    ),
                    "nullable": field.nullable,
                    "status": record.get(
                        "status",
                        "",
                    ),
                    "source_id": record.get(
                        "source_id",
                        "",
                    ),
                })

            # NOT NULL列の欠損も検出
            elif (
                not field.nullable
                and value is None
            ):
                invalid_log_values.append({
                    "row_index": row_index,
                    "column_name": field.name,
                    "data_type": str(
                        field.dataType
                    ),
                    "nullable": field.nullable,
                    "status": record.get(
                        "status",
                        "",
                    ),
                    "source_id": record.get(
                        "source_id",
                        "",
                    ),
                })

    if invalid_log_values:
        invalid_log_values_df = (
            spark.createDataFrame(
                invalid_log_values
            )
        )

        display(
            invalid_log_values_df
            .orderBy(
                "row_index",
                "column_name",
            )
        )

        raise ValueError(
            "Invalid None values were detected "
            "in log_records. "
            "Re-run Cell 11, Cell 13, and Cell 14 "
            "before saving."
        )

    print(
        "Validation passed: "
        "log_records contain no invalid None values"
    )

    # ---------------------------------------
    # Create DataFrame using existing schema
    # ---------------------------------------
    scrape_log_output_df = (
        spark.createDataFrame(
            log_records,
            schema=scrape_log_schema,
        )
    )

    print(
        "Scrape log rows prepared:",
        scrape_log_output_df.count(),
    )

    # ---------------------------------------
    # Display before writing
    # ---------------------------------------
    display(
        scrape_log_output_df
    )

    # ---------------------------------------
    # Append to Delta table
    # ---------------------------------------
    (
        scrape_log_output_df.write
        .format("delta")
        .mode("append")
        .saveAsTable(
            SCRAPE_LOG_TABLE
        )
    )

    print(
        "Inserted scrape log rows:",
        len(log_records),
    )

In [0]:
if log_records:
    scrape_log_output_df = (
        spark.createDataFrame(
            log_records,
            schema=scrape_log_schema,
        )
    )

    (
        scrape_log_output_df.write
        .format("delta")
        .mode("append")
        .saveAsTable(
            SCRAPE_LOG_TABLE
        )
    )

    print(
        "Inserted scrape log rows:",
        len(log_records),
    )

else:
    print(
        "No scrape log rows to insert."
    )
missing_required_values = []

for row_index, record in enumerate(
    log_records
):
    for field in scrape_log_schema.fields:
        if (
            not field.nullable
            and record.get(
                field.name
            ) is None
        ):
            missing_required_values.append({
                "row_index": row_index,
                "column_name": field.name,
                "data_type": str(
                    field.dataType
                ),
            })

if missing_required_values:
    display(
        spark.createDataFrame(
            missing_required_values
        )
    )

    raise ValueError(
        "Non-nullable scrape_log fields "
        "contain None"
    )

print(
    "Validation passed: "
    "all required scrape_log fields "
    "have values"
)

In [0]:
for field in scrape_log_schema.fields:
    print(
        field.name,
        field.dataType,
        "nullable=",
        field.nullable,
    )

In [0]:
for field in scrape_log_schema.fields:
    print(
        field.name,
        field.dataType,
        "nullable=",
        field.nullable,
    )



scrape_log_schema = (
    spark.table(
        SCRAPE_LOG_TABLE
    ).schema
)
if log_records:
    scrape_log_output_df = (
        spark.createDataFrame(
            log_records,
            schema=scrape_log_schema,
        )
    )

    (
        scrape_log_output_df.write
        .format("delta")
        .mode("append")
        .saveAsTable(
            SCRAPE_LOG_TABLE
        )
    )

    print(
        "Inserted scrape log rows:",
        len(log_records),
    )

In [0]:
run_finished_at = datetime.now(
    timezone.utc
)

run_elapsed_ms = int(
    (
        run_finished_at
        - run_started_at
    ).total_seconds()
    * 1000
)

print("=" * 60)
print("Tournament result ingestion completed")
print("=" * 60)

print("Run ID               :", run_id)
print("Targets              :", target_count)
print("Success              :", success_count)
print("Success empty        :", success_empty_count)
print("Skipped unchanged    :", skipped_count)
print("Error                :", error_count)
print("Inserted raw versions:", len(raw_records))
print("Elapsed milliseconds :", run_elapsed_ms)

In [0]:
if "run_id" in event_raw_columns:
    display(
        spark.table(
            EVENT_RESULT_RAW_TABLE
        )
        .filter(
            F.col("run_id")
            == run_id
        )
    )

In [0]:
if "run_id" in scrape_log_columns:
    display(
        spark.table(
            SCRAPE_LOG_TABLE
        )
        .filter(
            F.col("run_id")
            == run_id
        )
        .orderBy(
            F.col(
                "scraped_at"
            ).asc_nulls_last()
            if (
                "scraped_at"
                in scrape_log_columns
            )
            else F.lit(1)
        )
    )

In [0]:
if {
    "tournament_id",
    "response_hash",
}.issubset(
    event_raw_columns
):
    duplicate_versions_df = (
        spark.table(
            EVENT_RESULT_RAW_TABLE
        )
        .groupBy(
            "tournament_id",
            "response_hash",
        )
        .count()
        .filter(
            F.col("count") > 1
        )
    )

    duplicate_count = (
        duplicate_versions_df.count()
    )

    if duplicate_count > 0:
        display(
            duplicate_versions_df
        )

        raise ValueError(
            f"{duplicate_count} duplicate "
            "event result response versions "
            "detected"
        )

    print(
        "Validation passed: "
        "no duplicate tournament response "
        "versions"
    )

In [0]:
if (
    FAIL_NOTEBOOK_IF_ANY_ERROR
    and error_count > 0
):
    raise RuntimeError(
        "Some tournament result requests "
        "failed. "
        f"run_id={run_id}, "
        f"error_count={error_count}"
    )